In [1]:
%pip install -q diffusers transformers accelerate safetensors

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path

cwd = Path.cwd()
print("CWD:", cwd)


ROOT = None
for p in [cwd] + list(cwd.parents):
    if p.name == "Thermal_Depth_Sandbox":
        ROOT = p
        break

if ROOT is None:
    raise RuntimeError("Could not find Thermal_Depth_Sandbox in the parent path. Open the correct folder in VS Code.")

print("ROOT:", ROOT)

THERMAL_DIR = ROOT / "input (thermal)"
TADN_DIR = ROOT / "outputs_tadn_v3"
RAW_DEPTH_DIR = ROOT / "depth outputs"
RGB_OUT_DIR = ROOT / "rgb_outputs_tadn_v3"
RGB_OUT_DIR.mkdir(exist_ok=True)

print("THERMAL_DIR:", THERMAL_DIR, "exists:", THERMAL_DIR.exists())
print("TADN_DIR:", TADN_DIR, "exists:", TADN_DIR.exists())
print("RAW_DEPTH_DIR:", RAW_DEPTH_DIR, "exists:", RAW_DEPTH_DIR.exists())

CWD: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox
ROOT: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox
THERMAL_DIR: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input (thermal) exists: True
TADN_DIR: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_tadn_v3 exists: True
RAW_DEPTH_DIR: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth outputs exists: True


In [4]:
from PIL import Image

i = 1
thermal_path = THERMAL_DIR / f"t{i}.jpeg"
depth_path = TADN_DIR / f"t{i}_depth_tadn.png"

thermal = Image.open(thermal_path).convert("RGB")
control = Image.open(depth_path).convert("RGB")

target_size = (512, 512)
thermal = thermal.resize(target_size)
control = control.resize(target_size)

print("Loaded:", thermal_path.name, "and", depth_path.name)

Loaded: t1.jpeg and t1_depth_tadn.png


In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image
from transformers import pipeline

OUT_DIR = Path("/Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_PATH = Path("/Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input (thermal)/t1.jpeg")

pil_img = Image.open(IMG_PATH).convert("RGB")


depth_pipe = pipeline("depth-estimation", model="Intel/dpt-hybrid-midas", device=-1)  

pred = depth_pipe(pil_img)          
depth_img = pred["depth"]           

depth_np = np.array(depth_img)
print("Depth:", depth_np.shape, depth_np.min(), depth_np.max())

save_path = OUT_DIR / "t1_depth.png"
depth_img.save(save_path)
print("Saved:", save_path)

/Users/maryamellathy/Desktop/NEW thermaltoRBG/.venv_color/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Depth: (512, 640) 2 255
Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs/t1_depth.png


In [ ]:
import torch
from diffusers import ControlNetModel, StableDiffusionControlNetImg2ImgPipeline, UniPCMultistepScheduler

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

dtype = torch.float32 if device == "mps" else (torch.float16 if device == "cuda" else torch.float32)

print("Device:", device, "dtype:", dtype)

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=dtype
)

pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
)

pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)


pipe.enable_attention_slicing()
pipe.enable_vae_slicing()


pipe.vae.to(dtype=torch.float32)

Device: mps dtype: torch.float32


Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 17.74it/s]
/opt/homebrew/lib/python3.11/site-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionControlNetImg2ImgPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


AutoencoderKL(
  (encoder): Encoder(
    (conv_in): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (down_blocks): ModuleList(
      (0): DownEncoderBlock2D(
        (resnets): ModuleList(
          (0-1): 2 x ResnetBlock2D(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 128, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
          )
        )
        (downsamplers): ModuleList(
          (0): Downsample2D(
            (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(2, 2))
          )
        )
      )
      (1): DownEncoderBlock2D(
        (resnets): ModuleList(
          (0): ResnetBlock2D(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (c

In [ ]:
import numpy as np

thermal = pil_img          
control = depth_img        

print("thermal size:", thermal.size, "control size:", control.size)
print("thermal range:", np.array(thermal).min(), np.array(thermal).max())
print("control range:", np.array(control).min(), np.array(control).max())

thermal size: (640, 512) control size: (640, 512)
thermal range: 0 255
control range: 2 255


In [ ]:
import torch, types


if not hasattr(torch, "xpu"):
    torch.xpu = types.SimpleNamespace()

if not hasattr(torch.xpu, "is_available"):
    torch.xpu.is_available = lambda: False

if not hasattr(torch.xpu, "empty_cache"):
    torch.xpu.empty_cache = lambda: None

if not hasattr(torch.xpu, "device_count"):
    torch.xpu.device_count = lambda: 0

if not hasattr(torch.xpu, "manual_seed"):
    torch.xpu.manual_seed = torch.manual_seed

from diffusers import ControlNetModel, StableDiffusionControlNetImg2ImgPipeline, DPMSolverMultistepScheduler
print(" diffusers imported")

/Users/maryamellathy/Desktop/NEW thermaltoRBG/.venv_color/lib/python3.11/site-packages/torch/amp/autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


✅ diffusers imported


In [ ]:
import torch
from diffusers import ControlNetModel, StableDiffusionControlNetImg2ImgPipeline, DPMSolverMultistepScheduler

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Device:", device)


dtype = torch.float32

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=dtype
)

pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
)


pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

pipe = pipe.to(device)


pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.unet.to(dtype=torch.float32)
pipe.controlnet.to(dtype=torch.float32)
pipe.vae.to(dtype=torch.float32)

print("Pipeline ready ")

Device: mps


config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

...

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Pipeline ready ✅


/Users/maryamellathy/Desktop/NEW thermaltoRBG/.venv_color/lib/python3.11/site-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionControlNetImg2ImgPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


In [5]:
from pathlib import Path

cwd = Path.cwd()
print("CWD:", cwd)

ROOT = None
for p in [cwd] + list(cwd.parents):
    if p.name == "Thermal_Depth_Sandbox":
        ROOT = p
        break

if ROOT is None:
    raise RuntimeError("Thermal_Depth_Sandbox not found. Open the correct folder in VS Code.")

THERMAL_DIR = ROOT / "input"
RAW_DEPTH_DIR = ROOT / "outputs"
TADN_DIR = ROOT / "outputs_tadn_v3"
RGB_OUT_DIR = ROOT / "rgb_outputs_tadn_v3"
RGB_OUT_DIR.mkdir(exist_ok=True)

print("ROOT:", ROOT)
print("THERMAL_DIR exists:", THERMAL_DIR.exists())
print("TADN_DIR exists:", TADN_DIR.exists())
print("RAW_DEPTH_DIR exists:", RAW_DEPTH_DIR.exists())

CWD: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox
ROOT: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox
THERMAL_DIR exists: True
TADN_DIR exists: True
RAW_DEPTH_DIR exists: True


In [6]:
from PIL import Image

i = 1

thermal_path = THERMAL_DIR / f"t{i}.jpeg"
depth_path = TADN_DIR / f"t{i}_depth_tadn.png"

print("Thermal exists:", thermal_path.exists())
print("Depth exists:", depth_path.exists())

thermal = Image.open(thermal_path).convert("RGB")
control = Image.open(depth_path).convert("RGB")

target_size = (512, 512)
thermal = thermal.resize(target_size)
control = control.resize(target_size)

print("Thermal size:", thermal.size)
print("Control size:", control.size)

Thermal exists: True
Depth exists: True
Thermal size: (512, 512)
Control size: (512, 512)


In [12]:
prompt = "a realistic street scene, photo"
negative_prompt = "cartoon, painting, blurry"

generator = torch.Generator(device="cpu").manual_seed(0)

result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=thermal,
    control_image=control,
    strength=0.6,
    num_inference_steps=15,
    guidance_scale=7.0,
    controlnet_conditioning_scale=0.4,
    generator=generator
).images[0]

out_path = RGB_OUT_DIR / "t1_rgb_tadn_test.png"
result.save(out_path)
print("Saved:", out_path)

100%|██████████| 9/9 [13:30<00:00, 90.03s/it] 


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/rgb_outputs_tadn_v3/t1_rgb_tadn_test.png


In [ ]:
from PIL import Image

i = 1
target_size = (512, 512)

thermal = Image.open(THERMAL_DIR / f"t{i}.jpeg").convert("RGB").resize(target_size)

control_raw  = Image.open(RAW_DEPTH_DIR / f"t{i}_depth.png").convert("RGB").resize(target_size)
control_tadn = Image.open(TADN_DIR / f"t{i}_depth_tadn.png").convert("RGB").resize(target_size)

print("Loaded thermal + raw depth + TADN depth")

Loaded thermal + raw depth + TADN depth ✅


In [9]:
prompt = "a realistic street scene, photo"
negative_prompt = "cartoon, painting, blurry"

seed = 0
steps = 15
strength = 0.55
guidance = 5.5
c_scale = 0.7

In [10]:
import torch

gen_raw = torch.Generator(device="cpu").manual_seed(seed)

img_raw = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=thermal,
    control_image=control_raw,
    strength=strength,
    num_inference_steps=steps,
    guidance_scale=guidance,
    controlnet_conditioning_scale=c_scale,
    generator=gen_raw
).images[0]

out_raw = RGB_OUT_DIR / f"t{i}_rgb_rawdepth.png"
img_raw.save(out_raw)
print("Saved:", out_raw)

100%|██████████| 8/8 [09:31<00:00, 71.39s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/rgb_outputs_tadn_v3/t1_rgb_rawdepth.png


In [11]:
gen_tadn = torch.Generator(device="cpu").manual_seed(seed)

img_tadn = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=thermal,
    control_image=control_tadn,
    strength=strength,
    num_inference_steps=steps,
    guidance_scale=guidance,
    controlnet_conditioning_scale=c_scale,
    generator=gen_tadn
).images[0]

out_tadn = RGB_OUT_DIR / f"t{i}_rgb_tadn_v3.png"
img_tadn.save(out_tadn)
print("Saved:", out_tadn)

100%|██████████| 8/8 [09:56<00:00, 74.61s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/rgb_outputs_tadn_v3/t1_rgb_tadn_v3.png


In [13]:
import numpy as np
from PIL import Image

def stats(img, name):
    arr = np.array(img)
    print(name, "size:", img.size, "min/max:", arr.min(), arr.max(), "mean:", arr.mean())

stats(thermal, "thermal")
stats(control_raw, "control_raw")
stats(control_tadn, "control_tadn")

thermal size: (512, 512) min/max: 0 255 mean: 132.5492935180664
control_raw size: (512, 512) min/max: 0 255 mean: 79.31726837158203
control_tadn size: (512, 512) min/max: 2 255 mean: 93.8950309753418


In [14]:
import numpy as np
from PIL import Image

inv = Image.fromarray(255 - np.array(control_tadn))

In [ ]:
from PIL import Image

i = 1
target_size = (512, 512)

thermal = Image.open(THERMAL_DIR / f"t{i}.jpeg").convert("RGB").resize(target_size)
control_safe = Image.open(TADN_DIR / "t1_depth_tadn_safe.png").convert("RGB").resize(target_size)

print("Loaded thermal + control_safe", thermal.size, control_safe.size)

Loaded thermal + control_safe ✅ (512, 512) (512, 512)


In [20]:
import torch

prompt = "a realistic street scene, photo"
negative_prompt = "cartoon, painting, blurry"

seed = 0
steps = 15
strength = 0.55
guidance = 5.5
c_scale = 0.7

gen = torch.Generator(device="cpu").manual_seed(seed)

img_safe = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=thermal,
    control_image=control_safe,
    strength=strength,
    num_inference_steps=steps,
    guidance_scale=guidance,
    controlnet_conditioning_scale=c_scale,
    generator=gen
).images[0]

out_safe_rgb = RGB_OUT_DIR / "t1_rgb_tadn_safe.png"
img_safe.save(out_safe_rgb)
print("Saved:", out_safe_rgb)

100%|██████████| 8/8 [12:25<00:00, 93.22s/it] 


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/rgb_outputs_tadn_v3/t1_rgb_tadn_safe.png


In [23]:
import torch
print("MPS available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

print("pipe device:", pipe.device)
print("unet dtype:", pipe.unet.dtype)
print("vae dtype:", pipe.vae.dtype)
print("controlnet dtype:", pipe.controlnet.dtype)

MPS available: True
CUDA available: False
pipe device: mps:0
unet dtype: torch.float32
vae dtype: torch.float32
controlnet dtype: torch.float32


In [ ]:
import torch

prompt = (
    "photorealistic nighttime street scene, realistic colors, "
    "warm yellow street lights, green trees, grey asphalt road, "
    "color photograph, DSLR photo"
)

negative_prompt = (
    "black and white, grayscale, monochrome, desaturated, "
    "cartoon, painting, blurry"
)

seed = 0
steps = 15          
strength = 0.55     
guidance = 7.5      
c_scale = 0.7       

gen = torch.Generator(device="cpu").manual_seed(seed)

img_color_fixed = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=thermal,
    control_image=control_safe,
    strength=strength,
    num_inference_steps=steps,
    guidance_scale=guidance,
    controlnet_conditioning_scale=c_scale,
    generator=gen
).images[0]

out_path = RGB_OUT_DIR / "t1_rgb_tadn_safe_color_fixed.png"
img_color_fixed.save(out_path)
print("Saved:", out_path)

ValueError: Input is in incorrect format. Currently, we only support <class 'PIL.Image.Image'>, <class 'numpy.ndarray'>, <class 'torch.Tensor'>

In [4]:
from pathlib import Path

RGB_OUT_DIR = Path("/Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/rgb_outputs_tadn_v3")
RGB_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("RGB_OUT_DIR set to:", RGB_OUT_DIR)

RGB_OUT_DIR set to: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/rgb_outputs_tadn_v3


In [5]:
import numpy as np
from PIL import Image

p = RGB_OUT_DIR / "t1_rgb_tadn_safe_color_fixed.png"
img = Image.open(p).convert("RGB")
arr = np.array(img).astype(np.float32)

rg = np.mean(np.abs(arr[...,0] - arr[...,1]))
gb = np.mean(np.abs(arr[...,1] - arr[...,2]))
rb = np.mean(np.abs(arr[...,0] - arr[...,2]))

print("Mean |R-G|:", rg)
print("Mean |G-B|:", gb)
print("Mean |R-B|:", rb)

print(
    "Channel min/max:",
    "R", arr[...,0].min(), arr[...,0].max(),
    "G", arr[...,1].min(), arr[...,1].max(),
    "B", arr[...,2].min(), arr[...,2].max()
)

Mean |R-G|: 3.8368034
Mean |G-B|: 2.1475258
Mean |R-B|: 5.052391
Channel min/max: R 0.0 254.0 G 0.0 255.0 B 0.0 255.0


In [ ]:
import torch
from diffusers import StableDiffusionControlNetImg2ImgPipeline, ControlNetModel
from diffusers import DPMSolverMultistepScheduler

DEVICE = "mps"
DTYPE = torch.float32   

BASE_MODEL = "runwayml/stable-diffusion-v1-5"
CTRL_MODEL = "lllyasviel/sd-controlnet-depth"

controlnet = ControlNetModel.from_pretrained(
    CTRL_MODEL,
    torch_dtype=DTYPE
)

pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    BASE_MODEL,
    controlnet=controlnet,
    torch_dtype=DTYPE,
    safety_checker=None,
    requires_safety_checker=False
).to(DEVICE)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

print("pipe reloaded", pipe.device)

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/homebrew/lib/python3.11/site-packages/diffusers/models/transformers/transformer_kandinsky.py:168: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
/opt/homebrew/lib/python3.11/site-packages/diffusers/models/transformers/transformer_kandinsky.py:272: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 24.58it/s]


pipe reloaded ✅ mps:0


/opt/homebrew/lib/python3.11/site-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionControlNetImg2ImgPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


In [ ]:
from pathlib import Path
from PIL import Image


ROOT = Path("/Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox")


TADN_DIR = ROOT / "outputs_tadn_v3"
RGB_OUT_DIR = ROOT / "rgb_outputs_tadn_v3"

RGB_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("TADN_DIR exists?", TADN_DIR.exists())
print("RGB_OUT_DIR exists?", RGB_OUT_DIR.exists())

ROOT: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox
TADN_DIR exists? True
RGB_OUT_DIR exists? True


In [ ]:
from PIL import Image

control_safe = Image.open(TADN_DIR / "t1_depth_tadn_safe.png") \
    .convert("RGB") \
    .resize((512, 512))

print("control_safe loaded", control_safe.size, control_safe.mode)

control_safe loaded ✅ (512, 512) RGB


In [16]:
import torch
from PIL import Image

init = Image.open(RGB_OUT_DIR / "t1_rgb_tadn_safe_color_fixed.png") \
            .convert("RGB") \
            .resize((512,512))

prompt2 = (
    "photorealistic nighttime street scene, vivid realistic color photograph, "
    "warm yellow street lights, green trees, grey asphalt road, "
    "realistic color grading, DSLR photo"
)

neg2 = (
    "black and white, grayscale, monochrome, desaturated, "
    "cartoon, painting, blurry"
)

gen = torch.Generator(device="cpu").manual_seed(0)

img_color_lift = pipe(
    prompt=prompt2,
    negative_prompt=neg2,
    image=init,
    control_image=control_safe,
    strength=0.35,
    num_inference_steps=15,
    guidance_scale=10.0,
    controlnet_conditioning_scale=0.55,
    generator=gen
).images[0]

out2 = RGB_OUT_DIR / "t1_rgb_tadn_color_lift.png"
img_color_lift.save(out2)
print("Saved:", out2)

100%|██████████| 5/5 [06:36<00:00, 79.31s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/rgb_outputs_tadn_v3/t1_rgb_tadn_color_lift.png


In [ ]:
import torch

prompt = (
    "photorealistic nighttime street scene, vivid realistic color photograph, "
    "warm yellow street lights, green trees, grey asphalt road, "
    "natural colors, DSLR photo"
)
neg = "black and white, grayscale, monochrome, desaturated, noisy, oversharpened, cartoon, painting, blurry"

gen = torch.Generator(device="cpu").manual_seed(0)

img_col = pipe(
    prompt=prompt,
    negative_prompt=neg,
    image=init,                      
    control_image=control_safe,       
    strength=0.25,                    
    num_inference_steps=20,
    guidance_scale=8.5,
    controlnet_conditioning_scale=0.10,  
    generator=gen
).images[0]

out_path = RGB_OUT_DIR / "t1_rgb_color_only_noCN.png"
img_col.save(out_path)
print("Saved:", out_path)

100%|██████████| 5/5 [09:22<00:00, 112.42s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/rgb_outputs_tadn_v3/t1_rgb_color_only_noCN.png


In [ ]:
import gc, torch

# delete old pipes if they exist
for name in ["pipe", "pipe_txt", "pipe_color"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.mps.empty_cache()
print("Cleared pipes + emptied MPS cache")

Cleared pipes + emptied MPS cache ✅


In [ ]:
from PIL import Image

control = Image.open(TADN_DIR / "t1_depth_tadn_safe.png").convert("RGB")
control = control.resize((384, 384))  
print("control:", control.size)

control: (384, 384)


In [ ]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from diffusers import DPMSolverMultistepScheduler

DEVICE = "mps"
DTYPE = torch.float32  

BASE_MODEL = "runwayml/stable-diffusion-v1-5"
CTRL_MODEL = "lllyasviel/sd-controlnet-depth"

controlnet = ControlNetModel.from_pretrained(CTRL_MODEL, torch_dtype=DTYPE)

pipe_txt = StableDiffusionControlNetPipeline.from_pretrained(
    BASE_MODEL,
    controlnet=controlnet,
    torch_dtype=DTYPE,
    safety_checker=None,
    requires_safety_checker=False
).to(DEVICE)

pipe_txt.scheduler = DPMSolverMultistepScheduler.from_config(pipe_txt.scheduler.config)
pipe_txt.enable_attention_slicing()
pipe_txt.enable_vae_slicing()

print("Loaded pipe_txt", pipe_txt.device)

Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 13.12it/s]


Loaded pipe_txt ✅ mps:0


/opt/homebrew/lib/python3.11/site-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionControlNetPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


In [34]:
from PIL import Image
import numpy as np
from pathlib import Path

THERMAL_PATH = Path(
    "/Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input/t1.jpeg"
)

thermal = Image.open(THERMAL_PATH).convert("L").resize((512,512))

In [ ]:
from pathlib import Path
from PIL import Image

THERMAL_PATH = Path(
    "/Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input/t1.jpeg"
)

print("Exists:", THERMAL_PATH.exists())

thermal = Image.open(THERMAL_PATH).convert("L").resize((512, 512))
print("Loaded thermal", thermal.size)

Exists: True
Loaded thermal ✅ (512, 512)


In [ ]:
import numpy as np
from PIL import Image

arr = np.array(thermal).astype(np.float32)


r = arr * 1.00
g = arr * 0.99
b = arr * 0.97

init_rgb = Image.fromarray(
    np.stack([r, g, b], axis=-1).clip(0,255).astype(np.uint8),
    mode="RGB"
)

print("init_rgb:", init_rgb.size, init_rgb.mode)

init_rgb: (512, 512) RGB


In [39]:
from PIL import ImageFilter
init_rgb = init_rgb.filter(ImageFilter.GaussianBlur(radius=0.6))

In [ ]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, DDIMScheduler

DEVICE = "cpu"      
DTYPE = torch.float32

BASE_MODEL = "runwayml/stable-diffusion-v1-5"
CTRL_MODEL = "lllyasviel/sd-controlnet-depth"

print("Loading ControlNet…")
controlnet = ControlNetModel.from_pretrained(
    CTRL_MODEL,
    torch_dtype=DTYPE
)

print("Building pipeline…")
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    BASE_MODEL,
    controlnet=controlnet,
    torch_dtype=DTYPE,
    safety_checker=None,
    requires_safety_checker=False
)

pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.enable_attention_slicing()   
pipe = pipe.to(DEVICE)

print("Pipeline ready")

Loading ControlNet…
Building pipeline…


Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 14.62it/s]

Pipeline ready ✅


In [42]:
gen = torch.Generator(device="cpu").manual_seed(0)

img_col = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=init_rgb,
    control_image=control_safe,
    strength=0.22,
    num_inference_steps=28,
    guidance_scale=6.0,
    controlnet_conditioning_scale=0.35,
    generator=gen
).images[0]

out_path = RGB_OUT_DIR / "t1_rgb_colorised_grounded.png"
img_col.save(out_path)
print("Saved:", out_path)

100%|██████████| 28/28 [12:05<00:00, 25.90s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/rgb_outputs_tadn_v3/t1_rgb_colorised_grounded.png


In [ ]:
gen = torch.Generator(device="cpu").manual_seed(0)

img_col = pipe(
    prompt=(
        "same scene, same layout, minimal colourisation, "
        "natural muted colours, realistic lighting"
    ),
    negative_prompt=(
        "new buildings, different street, modern city, "
        "architectural changes, repainting, stylized, cinematic, "
        "oversaturated, vibrant, fantasy, illustration"
    ),
    image=init_rgb,                
    control_image=control_safe,     
    strength=0.12,                 
    num_inference_steps=30,
    guidance_scale=4.0,             
    controlnet_conditioning_scale=0.5,
    generator=gen
).images[0]

out_path = RGB_OUT_DIR / "t1_rgb_colorised_conservative.png"
img_col.save(out_path)
print("Saved:", out_path)

100%|██████████| 30/30 [13:10<00:00, 26.35s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/rgb_outputs_tadn_v3/t1_rgb_colorised_conservative.png


In [ ]:
gen = torch.Generator(device="cpu").manual_seed(0)

img_col = pipe(
    prompt=(
        "photorealistic nighttime street scene, realistic colors, "
        "warm yellow street lights, green trees, grey asphalt road, "
        "color photograph, DSLR photo, same scene, no scene change"
    ),
    negative_prompt=(
        "new buildings, different street, interior, garage, wall texture, "
        "modern architecture, cinematic, dramatic lighting, neon, "
        "oversaturated, fantasy, illustration, repainting, stylized, "
        "black and white, grayscale, monochrome, desaturated"
    ),
    image=init_rgb,                
    control_image=control_safe,     
    strength=0.10,                  
    num_inference_steps=32,         
    guidance_scale=3.5,             
    controlnet_conditioning_scale=0.6,
    generator=gen
).images[0]

out_path = RGB_OUT_DIR / "t1_rgb_colorised_locked.png"
img_col.save(out_path)
print("Saved:", out_path)

100%|██████████| 32/32 [14:11<00:00, 26.60s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/rgb_outputs_tadn_v3/t1_rgb_colorised_locked.png


In [ ]:
import torch
from diffusers import ControlNetModel, StableDiffusionControlNetImg2ImgPipeline, DPMSolverMultistepScheduler

device = "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float32  

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=dtype
)

pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

pipe.unet.to(dtype=torch.float32)
pipe.controlnet.to(dtype=torch.float32)
pipe.vae.to(dtype=torch.float32)

print("✅ pipe ready")

/opt/homebrew/lib/python3.11/site-packages/diffusers/models/transformers/transformer_kandinsky.py:168: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
/opt/homebrew/lib/python3.11/site-packages/diffusers/models/transformers/transformer_kandinsky.py:272: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
/opt/homebrew/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading weights: 100%|██████████| 196/196 [00:00<00:00, 258.70it/s, Materializing param=text_model.final_layer_norm.weight]
CLIPTextModel LOAD REPORT from: /Users/maryamellathy/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots

✅ pipe ready


/opt/homebrew/lib/python3.11/site-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionControlNetImg2ImgPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


In [10]:
from pathlib import Path

cwd = Path.cwd()

ROOT = None
for p in [cwd] + list(cwd.parents):
    if p.name == "Thermal_Depth_Sandbox":
        ROOT = p
        break

if ROOT is None:
    raise RuntimeError("Thermal_Depth_Sandbox not found. Open the correct folder.")

print("ROOT:", ROOT)

ROOT: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox


In [ ]:
from PIL import Image

init_rgb = Image.open(ROOT / "Input (thermal)" / "t1.jpeg").convert("RGB").resize((512,512))
control_safe = Image.open(ROOT / "outputs_tadn_v3" / "t1_depth_tadn_safe.png").convert("RGB").resize((512,512))

print("inputs loaded", init_rgb.size, control_safe.size)

✅ inputs loaded (512, 512) (512, 512)


In [15]:
from pathlib import Path

RGB_OUT_DIR = ROOT / "rgb_outputs_tadn_v3"
RGB_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("RGB_OUT_DIR:", RGB_OUT_DIR)

RGB_OUT_DIR: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/rgb_outputs_tadn_v3


In [ ]:
import torch

gen = torch.Generator(device="cpu").manual_seed(0)

prompt = (
    "photorealistic daytime suburban street intersection, strong sun glare and overexposed sky, "
    "hazy washed-out lighting, slight lens flare, "
    "downhill road stretching into the distance, asphalt road, "
    "white crosswalk lines and lane markings, "
    "trees lining both sides, utility poles and overhead power lines on the right, "
    "a white pickup truck parked on the right side, "
    "a car partially visible on the left edge, "
    "natural realistic colors, DSLR photo, same scene, no scene change"
)

negative_prompt = (
    "night, nighttime, street lights, warm yellow lamps, neon, rain, snow, foggy night, "
    "new buildings, different street, different intersection, different camera angle, "
    "interior, tunnel, garage, city skyline, downtown, modern architecture, "
    "cinematic dramatic lighting, high contrast, oversaturated, vibrant colors, "
    "cartoon, illustration, painting, stylized, anime, "
    "black and white, grayscale, monochrome, desaturated, "
    "extra vehicles, crowds, pedestrians, text, watermark"
)

img_col = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=init_rgb,              
    control_image=control_safe,  
    strength=0.10,               
    num_inference_steps=32,
    guidance_scale=3.5,          
    controlnet_conditioning_scale=0.6,
    generator=gen
).images[0]

out_path = RGB_OUT_DIR / "t1_rgb_colorised_locked.png"
img_col.save(out_path)
print("Saved:", out_path)

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['left edge , natural realistic colors , dslr photo , same scene , no scene change']
100%|██████████| 3/3 [06:45<00:00, 135.01s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/rgb_outputs_tadn_v3/t1_rgb_colorised_locked.png


In [ ]:
%pip -q install --upgrade diffusers==0.36.0 transformers==4.38.2 accelerate safetensors pillow numpy

from pathlib import Path
from PIL import Image
import numpy as np
import torch
from diffusers import ControlNetModel, StableDiffusionControlNetImg2ImgPipeline, DPMSolverMultistepScheduler


cwd = Path.cwd()
ROOT = None
for p in [cwd] + list(cwd.parents):
    if p.name == "Thermal_Depth_Sandbox":
        ROOT = p
        break

if ROOT is None:
    raise RuntimeError("Could not find folder 'Thermal_Depth_Sandbox'. Open that folder in VS Code and rerun.")

print("ROOT:", ROOT)


THERMAL_DIR_CANDIDATES = [ROOT / "input (thermal)", ROOT / "input"]
THERMAL_DIR = next((d for d in THERMAL_DIR_CANDIDATES if d.exists()), None)
if THERMAL_DIR is None:
    raise RuntimeError(f"Could not find thermal input folder. Tried: {THERMAL_DIR_CANDIDATES}")

TADN_DIR = ROOT / "outputs_tadn_v3"        
RGB_OUT_DIR = ROOT / "rgb_outputs_tadn_v3" 
RGB_OUT_DIR.mkdir(exist_ok=True)


i = 1

thermal_path = THERMAL_DIR / f"t{i}.jpeg"
control_path = TADN_DIR / f"t{i}_depth_tadn_safe.png"  

if not thermal_path.exists():
    raise FileNotFoundError(f"Thermal not found: {thermal_path}")
if not control_path.exists():
    raise FileNotFoundError(f"Control/TADN safe depth not found: {control_path}")

print("Thermal:", thermal_path)
print("Control:", control_path)
print("Output dir:", RGB_OUT_DIR)

target_size = (512, 512)
init_rgb = Image.open(thermal_path).convert("RGB").resize(target_size)
control_safe = Image.open(control_path).convert("RGB").resize(target_size)

print("init_rgb size:", init_rgb.size, "control size:", control_safe.size)


device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32 if device == "mps" else (torch.float16 if device == "cuda" else torch.float32)
print("Device:", device, "dtype:", dtype)

controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-depth", torch_dtype=dtype)

pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False,
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)


pipe.enable_attention_slicing()
pipe.enable_vae_slicing()


if device == "mps":
    pipe.unet.to(dtype=torch.float32)
    pipe.controlnet.to(dtype=torch.float32)
    pipe.vae.to(dtype=torch.float32)

print(" Pipeline ready")


prompt = (
    "photorealistic daytime suburban road, dashcam viewpoint looking straight down the street, "
    "trees lining both sides, utility poles and overhead power lines, "
    "a white pickup truck parked on the right shoulder, distant cars ahead, "
    "asphalt road with faint lane markings and crosswalk markings, "
    "strong sun glare from the horizon, slightly hazy/washed-out highlights, "
    "natural realistic colors, DSLR photo, ultra realistic, same scene, same camera angle"
)

negative_prompt = (
    "night, nighttime, street lamps, neon, rain, snow, foggy night, "
    "different road, different intersection, different camera angle, "
    "new buildings, city downtown, extra lanes, different vehicle layout, "
    "cartoon, painting, illustration, CGI, anime, stylized, "
    "oversaturated, dramatic cinematic lighting, unreal colors, "
    "text, watermark, logo"
)


seed = 0
gen = torch.Generator(device="cpu").manual_seed(seed)

img_col = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    image=init_rgb,                 
    control_image=control_safe,      
    strength=0.28,                   
    num_inference_steps=30,
    guidance_scale=4.5,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = RGB_OUT_DIR / f"t{i}_rgb_prompt_driven.png"
img_col.save(out_path)

print("Saved:", out_path)


arr = np.array(img_col)
print("RGB means:", arr[...,0].mean(), arr[...,1].mean(), arr[...,2].mean())
print("RGB stds: ", arr[...,0].std(),  arr[...,1].std(),  arr[...,2].std())


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ ROOT: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox
✅ Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input (thermal)/t1.jpeg
✅ Control: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_tadn_v3/t1_depth_tadn_safe.png
✅ Output dir: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/rgb_outputs_tadn_v3
init_rgb size: (512, 512) control size: (512, 512)
✅ Device: mps dtype: torch.float32


/opt/homebrew/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading weights: 100%|██████████| 196/196 [00:00<00:00, 298.68it/s, Materializing param=text_model.final_layer_norm.weight]
CLIPTextModel LOAD REPORT from: /Users/maryamellathy/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  5.15it/s]
/opt/homebrew/lib/python3.11/site-packages/diffusers/pipelines/pipelin

✅ Pipeline ready


100%|██████████| 8/8 [12:27<00:00, 93.41s/it] 


✅ Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/rgb_outputs_tadn_v3/t1_rgb_prompt_driven.png
RGB means: 131.89677047729492 130.803466796875 131.0315055847168
RGB stds:  60.14136387148719 61.07247529745198 60.518303452573505


In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from pathlib import Path


device = "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float32  

OUT_DIR = Path("rgb_free_generation")
OUT_DIR.mkdir(exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.vae.to(dtype=torch.float32)

print("RGB generation pipeline ready")


prompt = (
    "plain daytime suburban street, neutral lighting, "
    "overexposed sky, slight haze, washed out colors, "
    "ordinary residential road, asphalt street, trees along the road, "
    "parked vehicles, utility poles and power lines, "
    "traffic markings visible, "
    "realistic camera photo, unedited, no enhancement"
)

negative_prompt = (
    "artistic, stylized, cinematic, vibrant colors, saturated, "
    "lush greenery, dramatic lighting, golden hour, HDR, "
    "illustration, painting, render, CGI, unreal engine, "
    "stock photo, professional photography, aesthetic"
)

gen = torch.Generator(device="cpu").manual_seed(42)


img = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=40,
    guidance_scale=8.5,
    generator=gen
).images[0]

out_path = OUT_DIR / "rgb_free_generation.png"
img.save(out_path)

print("🎉 Saved:", out_path)

Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00,  6.11it/s]


KeyboardInterrupt: 

In [ ]:
import torch
from diffusers import (
    StableDiffusionControlNetImg2ImgPipeline,
    ControlNetModel,
    DPMSolverMultistepScheduler
)

device = "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float32   


controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=dtype
)

pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)


pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.vae.to(dtype=torch.float32)

print("ControlNet pipeline ready:", device)

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/homebrew/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/homebrew/lib/python3.11/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/opt/homebrew/lib/python3.11/site-packages/diffusers/models/transformers/transformer_kandinsky.py:168: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
/opt/homebrew/lib/python3.11/sit

✅ ControlNet pipeline ready: mps


/opt/homebrew/lib/python3.11/site-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionControlNetImg2ImgPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


In [ ]:
import torch
from diffusers import ControlNetModel, StableDiffusionControlNetImg2ImgPipeline, DPMSolverMultistepScheduler

device = "mps" if torch.backends.mps.is_available() else "cpu"

dtype = torch.float16 if device == "mps" else torch.float32

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=dtype
)

pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

pipe = pipe.to(device)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()


pipe.vae.to(dtype=torch.float32)

print("Pipeline ready on", device, "dtype:", dtype)

Loading pipeline components...: 100%|██████████| 6/6 [00:10<00:00,  1.83s/it]


✅ Pipeline ready on mps dtype: torch.float16


In [ ]:
import os
import gc
from pathlib import Path
import torch
from PIL import Image
from diffusers import ControlNetModel, StableDiffusionControlNetImg2ImgPipeline, DPMSolverMultistepScheduler

ROOT = Path("/Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox")

THERMAL_PATH   = ROOT / "input (thermal)" / "t2.jpeg"
DEPTH_SAFE_PATH = ROOT / "outputs_tadn_v3" / "t2_depth_tadn_safe.png"  
OUT_DIR        = ROOT / "rgb_free_generation"
OUT_DIR.mkdir(exist_ok=True)

assert THERMAL_PATH.exists(), f"Thermal not found: {THERMAL_PATH}"
assert DEPTH_SAFE_PATH.exists(), f"Safe depth not found: {DEPTH_SAFE_PATH}"

TARGET_SIZE = (384, 384)      
STEPS = 14                   
GUIDANCE = 6.0               
STRENGTH = 0.70              
C_SCALE = 0.80               


prompt = (
    "photorealistic full-color daytime road scene, natural RGB color photo, "
    "bright midday sunlight, pale blue sky, grey asphalt road, white lane markings, "
    "trees and bushes along the roadside, utility poles and power lines, "
    "realistic camera photograph, neutral daylight white balance, no scene change"
)

negative_prompt = (
    "black and white, grayscale, monochrome, infrared, thermal, night, darkness, "
    "cinematic, dramatic lighting, neon, HDR, oversaturated, stylized, artistic, "
    "illustration, painting, CGI, unreal engine, "
    "different scene, different road, different camera angle, text, watermark"
)


init_rgb = Image.open(THERMAL_PATH).convert("RGB").resize(TARGET_SIZE)

control_safe = Image.open(DEPTH_SAFE_PATH).convert("L").resize(TARGET_SIZE).convert("RGB")


def cleanup():
    gc.collect()
    if torch.backends.mps.is_available():
        try:
            torch.mps.empty_cache()
        except Exception:
            pass

cleanup()

USE_MPS = torch.backends.mps.is_available()
device = "mps" if USE_MPS else "cpu"
dtype = torch.float32  

print("Device:", device, "| dtype:", dtype, "| size:", TARGET_SIZE)


def build_pipe(device: str):
    controlnet = ControlNetModel.from_pretrained(
        "lllyasviel/sd-controlnet-depth",
        torch_dtype=dtype
    )

    pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        controlnet=controlnet,
        torch_dtype=dtype,
        safety_checker=None,
        requires_safety_checker=False
    )

    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

    pipe.enable_attention_slicing()
    pipe.enable_vae_slicing()

    pipe = pipe.to(device)

    
    pipe.unet.to(dtype=torch.float32)
    pipe.controlnet.to(dtype=torch.float32)
    pipe.vae.to(dtype=torch.float32)

    return pipe

try:
    pipe = build_pipe(device)
except RuntimeError as e:
    print("\n⚠️ MPS failed (likely OOM). Falling back to CPU...\n", str(e)[:400], "\n")
    cleanup()
    device = "cpu"
    pipe = build_pipe(device)

print("Pipeline ready on", device)


gen = torch.Generator(device="cpu").manual_seed(42)

with torch.inference_mode():
    out = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=init_rgb,
        control_image=control_safe,
        strength=STRENGTH,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        controlnet_conditioning_scale=C_SCALE,
        generator=gen,
    ).images[0]

out_path = OUT_DIR / "t2_rgb_image.png"
out.save(out_path)
print("🎉 Saved:", out_path)

Device: mps | dtype: torch.float32 | size: (384, 384)


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: bebbf00a-a51e-4682-a9a0-39a47309a2df)')' thrown while requesting HEAD https://huggingface.co/lllyasviel/sd-controlnet-depth/resolve/main/config.json
Retrying in 1s [Retry 1/5].
Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 17.25it/s]



⚠️ MPS failed (likely OOM). Falling back to CPU...
 MPS backend out of memory (MPS allocated: 9.03 GiB, other allocations: 2.61 MiB, max allowed: 9.07 GiB). Tried to allocate 50.00 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure). 



Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 10.04it/s]


✅ Pipeline ready on cpu


100%|██████████| 9/9 [03:05<00:00, 20.56s/it]


🎉 Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/rgb_free_generation/t2_rgb_image.png


In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from pathlib import Path

device = "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float32  

OUT_DIR = Path("rgb_free_generation")
OUT_DIR.mkdir(exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.vae.to(dtype=torch.float32)

print("RGB generation pipeline ready")

prompt = (
    "photorealistic full-color RGB daytime suburban road, early morning sunlight, "
    "very bright white sky with strong glare, slight haze, subtle lens flare, "
    "grey asphalt road, visible road texture, white lane markings, "
    "trees lining both left and right sides of the road, green foliage, "
    "utility poles and power lines along the right side, "
    "a few distant cars ahead on the road, "
    "realistic camera photograph, neutral daylight white balance, "
    "natural colors (blue sky tint, green trees, grey road), not stylized"
)

negative_prompt = (
    "black and white, grayscale, monochrome, desaturated, low saturation, "
    "infrared, thermal, night, nighttime, dusk, sunset, streetlights glowing, "
    "cinematic, dramatic lighting, moody, foggy night, snow, rain, "
    "cartoon, anime, illustration, painting, CGI, render, unreal engine, "
    "HDR, oversharpened, oversaturated, neon colors, "
    "text, watermark, logo"
)


gen = torch.Generator(device="cpu").manual_seed(123)   
steps = 24                                             
guidance = 7.5                                         

img = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=steps,
    guidance_scale=guidance,
    generator=gen
).images[0]

out_path = OUT_DIR / "t2_like_daytime_rgb.png"
img.save(out_path)
print("Saved:", out_path)

Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  5.98it/s]
/opt/homebrew/lib/python3.11/site-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(
Token indices sequence length is longer than the specified maximum sequence length for this model (101 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['camera photograph, neutral daylight white balance, natural colors ( blue sky tint, green trees, grey road ), not stylized']


✅ RGB generation pipeline ready


100%|██████████| 24/24 [29:59<00:00, 74.98s/it]


🎉 Saved: rgb_free_generation/t2_like_daytime_rgb.png


In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from pathlib import Path

device = "mps" if torch.backends.mps.is_available() else "cpu"

dtype = torch.float32

OUT_DIR = Path("rgb_free_generation")
OUT_DIR.mkdir(exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

pipe.enable_sequential_cpu_offload()

print("RGB generation pipeline ready (with sequential CPU offload)")

prompt = (
    "Photorealistic early morning suburban street scene with bright white sky"
    "and strong sun glare. Grey asphalt road with white lane markings,"
    "slightly overexposed surface. Cars driving and parked along the left side,"
    "one vehicle approaching from the distance. A public bench on the right sidewalk"
    "near residential houses. Tall green trees on both sides, utility poles"
    "and overhead power lines visible. Natural daylight, real camera photograph, unedited."
)

negative_prompt = (
    "artistic, illustration, cartoon, CGI, render, stylized,"
    "cinematic, dramatic lighting, HDR, saturated colors, neon, anime,"
    "painting, unreal engine, fantasy, fog, night, dusk, snow,"
    "rain, monochrome, grayscale"
)

gen = torch.Generator(device="cpu").manual_seed(123)

steps = 14          
guidance = 5.5      
height = 384        
width  = 384

img = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=steps,
    guidance_scale=guidance,
    height=height,
    width=width,
    generator=gen
).images[0]

out_path = OUT_DIR / "t3_rgb_free_generation_384.png"
img.save(out_path)

print("Saved:", out_path)

Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  3.88it/s]
/opt/homebrew/lib/python3.11/site-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(
Token indices sequence length is longer than the specified maximum sequence length for this model (86 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['daylight, real camera photograph, unedited.']


✅ RGB generation pipeline ready (with sequential CPU offload)


100%|██████████| 14/14 [11:11<00:00, 48.00s/it]


🎉 Saved: rgb_free_generation/t3_rgb_free_generation_384.png


In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from pathlib import Path

device = "mps" if torch.backends.mps.is_available() else "cpu"

dtype = torch.float32

OUT_DIR = Path("rgb_free_generation")
OUT_DIR.mkdir(exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()


pipe.enable_sequential_cpu_offload()

print("RGB generation pipeline ready (with sequential CPU offload)")


prompt = (
    "photorealistic early morning suburban street scene viewed from a dashcam,"
    "soft natural daylight with pale blue sky, slightly hazy atmosphere,"
    "gray asphalt road with visible cracks and texture, a single car driving ahead"
    "in the center of the road, low residential walls and houses on both sides,"
    "tall trees lining the street, mountains visible in the distance, utility poles"
    "and overhead power lines, calm quiet neighborhood, natural muted colors, realistic unedited camera photograph"
)

negative_prompt = (
    "artistic, illustration, cartoon, CGI, render, stylized,"
    "cinematic, dramatic lighting, HDR, saturated colors, neon, anime,"
    "painting, unreal engine, fantasy, fog, night, dusk, snow,"
    "rain, monochrome, grayscale"
)

gen = torch.Generator(device="cpu").manual_seed(123)


steps = 14          
guidance = 5.5      
height = 384        
width  = 384


img = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=steps,
    guidance_scale=guidance,
    height=height,
    width=width,
    generator=gen
).images[0]

out_path = OUT_DIR / "t5_generation.png"
img.save(out_path)

print("Saved:", out_path)

Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  5.50it/s]
/opt/homebrew/lib/python3.11/site-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(
Token indices sequence length is longer than the specified maximum sequence length for this model (96 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['and overheadpower lines, soft lens flare, natural daylight colors, realistic unedited camera photograph']


✅ RGB generation pipeline ready (with sequential CPU offload)


100%|██████████| 14/14 [01:21<00:00,  5.80s/it]


🎉 Saved: rgb_free_generation/t4_generation.png


In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from pathlib import Path


device = "mps" if torch.backends.mps.is_available() else "cpu"

dtype = torch.float32

OUT_DIR = Path("rgb_free_generation")
OUT_DIR.mkdir(exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

pipe.enable_sequential_cpu_offload()

print("RGB generation pipeline ready (with sequential CPU offload)")

prompt = (
    "plain daytime suburban intersection, neutral lighting, "
    "extremely overexposed sky with strong sun glare, heavy haze, washed out colors, "
    "ordinary residential road at a stop, asphalt street with large white STOP marking, "
    "rear of a white SUV stopped close to the camera on the right side, "
    "another car entering from the left side of the intersection, "
    "trees and houses in the background, "
    "utility poles and overhead power lines, "
    "traffic markings visible, "
    "realistic dashcam photo, unedited, no enhancement"
)

negative_prompt = (
    "artistic, stylized, cinematic, vibrant colors, saturated, "
    "dramatic lighting, golden hour, HDR, "
    "illustration, painting, render, CGI, unreal engine, "
    "aesthetic, clean sky, sharp contrast"
)

gen = torch.Generator(device="cpu").manual_seed(123)

steps = 14          
guidance = 5.5      
height = 384        
width  = 384

img = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=steps,
    guidance_scale=guidance,
    height=height,
    width=width,
    generator=gen
).images[0]

out_path = OUT_DIR / "t4_generation.png"
img.save(out_path)

print("Saved:", out_path)

Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00,  7.18it/s]
Token indices sequence length is longer than the specified maximum sequence length for this model (96 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['and overhead power lines, traffic markings visible, realistic dashcam photo, unedited, no enhancement']


✅ RGB generation pipeline ready (with sequential CPU offload)


100%|██████████| 14/14 [02:30<00:00, 10.76s/it]


🎉 Saved: rgb_free_generation/t4_generation.png


In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from pathlib import Path

device = "mps" if torch.backends.mps.is_available() else "cpu"

dtype = torch.float32

OUT_DIR = Path("rgb_free_generation")
OUT_DIR.mkdir(exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

pipe.enable_sequential_cpu_offload()

print("RGB generation pipeline ready (with sequential CPU offload)")

prompt = (
    "plain daytime suburban street, neutral lighting,"
    "slightly overexposed sky, light haze, washed out colors,"
    "wide residential road viewed from a distant dashcam perspective,"
    "gray asphalt street with faint lane markings,"
    "a single car driving far ahead near the center of the road,"
    "trees and hedges lining both sides,"
    "detached houses set back from the street,"
    "utility poles and overhead power lines,"
    "palm trees visible along the roadside,"
    "mountains faintly visible in the background,"
    "realistic camera photo, unedited, no enhancement"
)

negative_prompt = (
    "artistic, stylized, cinematic, vibrant colors, saturated,"
    "dramatic lighting, golden hour, HDR,"
    "illustration, painting, render, CGI, unreal engine,"
    "close-up view, low angle, exaggerated perspective"
)

gen = torch.Generator(device="cpu").manual_seed(123)

steps = 14          
guidance = 5.5      
height = 384        
width  = 384

img = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=steps,
    guidance_scale=guidance,
    height=height,
    width=width,
    generator=gen
).images[0]

out_path = OUT_DIR / "t6_generation.png"
img.save(out_path)

print("Saved:", out_path)

Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  5.06it/s]
/opt/homebrew/lib/python3.11/site-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(
Token indices sequence length is longer than the specified maximum sequence length for this model (102 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['palm trees visible along the roadside, mountains faintly visible in the background, realistic camera photo, unedited, no enhancement']


✅ RGB generation pipeline ready (with sequential CPU offload)


100%|██████████| 14/14 [01:35<00:00,  6.85s/it]


🎉 Saved: rgb_free_generation/t6_generation.png


In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from pathlib import Path

device = "mps" if torch.backends.mps.is_available() else "cpu"

dtype = torch.float32

OUT_DIR = Path("rgb_free_generation")
OUT_DIR.mkdir(exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)


pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

pipe.enable_sequential_cpu_offload()

print("RGB generation pipeline ready (with sequential CPU offload)")


prompt = (
    "plain daytime suburban road viewed from a distant dashcam perspective, "
    "wide straight asphalt street extending far into the distance, "
    "faint center lane markings visible, "
    "a single small car driving far ahead near the center of the road, "
    "no nearby vehicles in the foreground, "
    "trees and hedges lining both sides of the street, "
    "palm trees spaced along the roadside, "
    "residential houses set back from the road, "
    "utility poles and overhead power lines, "
    "mountains visible far in the background under a pale blue sky, "
    "slightly overexposed daylight, light haze, washed-out colors, "
    "realistic unedited dashcam photograph"
)

negative_prompt = (
    "close-up view, foreground vehicle, rear of car near camera, "
    "intersection, stop sign, traffic lights, parked van nearby, "
    "artistic, stylized, cinematic, dramatic lighting, HDR, "
    "vibrant colors, saturated colors, illustration, painting, "
    "CGI, render, unreal engine, fantasy scenery"
)

gen = torch.Generator(device="cpu").manual_seed(123)

steps = 14          
guidance = 5.5      
height = 384        
width  = 384

img = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=steps,
    guidance_scale=guidance,
    height=height,
    width=width,
    generator=gen
).images[0]

out_path = OUT_DIR / "t7_generation.png"
img.save(out_path)

print("Saved:", out_path)

Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  5.18it/s]
Token indices sequence length is longer than the specified maximum sequence length for this model (116 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['utility poles and overhead power lines, mountains visible far in the background under a pale blue sky, slightly overexposed daylight, light haze, washed - out colors, realistic unedited dashcam photograph']


✅ RGB generation pipeline ready (with sequential CPU offload)


100%|██████████| 14/14 [02:07<00:00,  9.09s/it]


🎉 Saved: rgb_free_generation/t7_generation.png


In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from pathlib import Path

device = "mps" if torch.backends.mps.is_available() else "cpu"

dtype = torch.float32

OUT_DIR = Path("rgb_free_generation")
OUT_DIR.mkdir(exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)


pipe.enable_attention_slicing()
pipe.enable_vae_slicing()


pipe.enable_sequential_cpu_offload()

print("RGB generation pipeline ready (with sequential CPU offload)")


prompt = (
    "realistic daytime dashcam photo of a quiet suburban street, "
    "wide gray asphalt road dominating the foreground, slightly curving left, "
    "soft overexposed sunlight creating haze and washed out colors, "
    "large leafy trees and tall evergreens lining the left side of the street, "
    "a low mountain ridge visible in the far background under a pale blue sky, "
    "on the right side a single-story suburban home set back from the road, "
    "visible driveway entrance and small front yard with shrubs and stone edging, "
    "subtle shadows on the pavement, natural unedited camera image"
)

negative_prompt = (
    "dramatic lighting, golden hour, sunset, HDR, vibrant colors, high contrast, "
    "luxury mansion, modern architecture, centered house, close-up building, "
    "cinematic composition, artistic photography, illustration, CGI, render"
)

gen = torch.Generator(device="cpu").manual_seed(123)


steps = 14          
guidance = 5.5      
height = 384        
width  = 384

img = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=steps,
    guidance_scale=guidance,
    height=height,
    width=width,
    generator=gen
).images[0]

out_path = OUT_DIR / "t9_generation.png"
img.save(out_path)

print("Saved:", out_path)

Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  4.26it/s]
Token indices sequence length is longer than the specified maximum sequence length for this model (109 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['home set back from the road, visible driveway entrance and small front yard with shrubs and stone edging, subtle shadows on the pavement, natural unedited camera image']


✅ RGB generation pipeline ready (with sequential CPU offload)


100%|██████████| 14/14 [03:11<00:00, 13.68s/it]


🎉 Saved: rgb_free_generation/t9_generation.png


In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from pathlib import Path

device = "mps" if torch.backends.mps.is_available() else "cpu"

dtype = torch.float32

OUT_DIR = Path("rgb_free_generation")
OUT_DIR.mkdir(exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

pipe.enable_sequential_cpu_offload()

print(" RGB generation pipeline ready (with sequential CPU offload)")


prompt = (
    "plain daytime suburban street, neutral daylight, "
    "long straight asphalt road with visible texture and center lane marking, "
    "trees evenly spaced on both sides forming a corridor, "
    "a single ordinary sedan parked clearly on the right side near the curb, "
    "a visible public bench placed on the right sidewalk under the trees, "
    "no people, no traffic, empty road except one parked car, "
    "slightly soft lighting, natural colors, realistic dashcam photo, "
    "unedited, no enhancement, no stylization"
)

negative_prompt = (
    "empty road with no vehicles, missing car, missing bench, "
    "artistic, cinematic, dramatic lighting, golden hour, HDR, "
    "illustration, painting, cartoon, anime, "
    "render, CGI, unreal engine, 3d, low poly, "
    "perfect symmetry, futuristic street, luxury cars"
)

gen = torch.Generator(device="cpu").manual_seed(123)


steps = 14          
guidance = 5.5      
height = 384        
width  = 384


img = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=steps,
    guidance_scale=guidance,
    height=height,
    width=width,
    generator=gen
).images[0]

out_path = OUT_DIR / "t3_generation.png"
img.save(out_path)

print("Saved:", out_path)

Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  3.96it/s]
Token indices sequence length is longer than the specified maximum sequence length for this model (94 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['natural colors, realistic dashcam photo, unedited, no enhancement, no stylization']


✅ RGB generation pipeline ready (with sequential CPU offload)


100%|██████████| 14/14 [02:25<00:00, 10.41s/it]


🎉 Saved: rgb_free_generation/t3_generation.png
